In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 2.1 Introduction to Regularization

- In Course 2: logistic regression with `penalty=None`
- Now: add regularization to prevent overfitting and select features
- This notebook: concepts. Next notebooks: implementation.

## What is Overfitting?

- Model learns training data **too well** — including noise
- Performs great on training data, poorly on new data
- Like memorizing practice exams but failing the real one

## Let's Visualize Overfitting

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)
X = np.linspace(0, 10, 20)
y_true = 2 * X + 1
y_noisy = y_true + np.random.normal(0, 3, len(X))

X_smooth = np.linspace(0, 10, 100)
y_underfit = np.ones_like(X_smooth) * np.mean(y_noisy)
coeffs_good = np.polyfit(X, y_noisy, 1)
y_goodfit = np.polyval(coeffs_good, X_smooth)
coeffs_overfit = np.polyfit(X, y_noisy, 15)
y_overfit = np.polyval(coeffs_overfit, X_smooth)

fig = make_subplots(rows=1, cols=3, subplot_titles=('Underfitting', 'Good Fit', 'Overfitting'))

for col in [1, 2, 3]:
    fig.add_trace(go.Scatter(x=X, y=y_noisy, mode='markers', name='Data', 
                             marker=dict(color='blue', size=8), showlegend=(col==1)), row=1, col=col)

fig.add_trace(go.Scatter(x=X_smooth, y=y_underfit, mode='lines', name='Model', 
                         line=dict(color='red', width=2), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=X_smooth, y=y_goodfit, mode='lines', name='Model', 
                         line=dict(color='green', width=2), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=X_smooth, y=y_overfit, mode='lines', name='Model', 
                         line=dict(color='red', width=2), showlegend=False), row=1, col=3)

fig.update_layout(height=400, title_text="The Fitting Spectrum: From Underfitting to Overfitting")
fig.show()

## Reading the Chart

- **Underfitting** — too simple, misses the trend
- **Good Fit** — captures the pattern, ignores noise
- **Overfitting** — follows every point, won't generalize

## Detecting Overfitting

| Scenario | Train Performance | Test Performance | Diagnosis |
|:---------|:-----------------|:-----------------|:----------|
| Good fit | High | High | Generalizes well |
| Overfitting | Very High | Low | Memorized training data |
| Underfitting | Low | Low | Too simple |

## Regularization: The Solution

- Adds a **penalty** to the loss function for large coefficients
- Overfitted models tend to have large coefficients
- Penalizing them → simpler models that generalize better

**Without regularization:** Loss = Prediction Error

**With regularization:** Loss = Prediction Error + λ × Penalty

## The λ (Lambda) Parameter

- λ = 0 → No regularization (Course 2)
- Small λ → Weak regularization
- Large λ → Strong regularization (simpler model)

> In scikit-learn: `C` = 1/λ (inverse!)
> Small C = strong regularization, Large C = weak regularization

## The Bias-Variance Trade-off

- **Bias**: Error from oversimplification → underfitting
- **Variance**: Error from sensitivity to training data → overfitting
- **Total Error** = Bias² + Variance + Irreducible Noise
- Regularization: slightly more bias, much less variance → better overall

## Let's Visualize the Trade-off

In [ ]:
complexity = np.linspace(0.1, 3, 100)
bias_squared = 1 / complexity
variance = 0.3 * complexity ** 2
total_error = bias_squared + variance + 0.1

fig = go.Figure()
fig.add_trace(go.Scatter(x=complexity, y=bias_squared, mode='lines', name='Bias²', 
                         line=dict(color='blue', width=2)))
fig.add_trace(go.Scatter(x=complexity, y=variance, mode='lines', name='Variance', 
                         line=dict(color='orange', width=2)))
fig.add_trace(go.Scatter(x=complexity, y=total_error, mode='lines', name='Total Error', 
                         line=dict(color='red', width=3)))

optimal_idx = np.argmin(total_error)
fig.add_vline(x=complexity[optimal_idx], line_dash="dash", line_color="green", 
              annotation_text="Optimal Complexity")

fig.update_layout(title='The Bias-Variance Trade-off', xaxis_title='Model Complexity',
                  yaxis_title='Error', height=400, showlegend=True)
fig.show()

## Three Types of Regularization

## L2 Regularization (Ridge)

- Penalty = sum of **squared** coefficients
- Shrinks all coefficients toward zero
- **Never** zeros them out completely
- Keeps all features in the model
- Best when many features contribute

> *Like putting coefficients on a diet — everyone loses weight, nobody disappears*

## L1 Regularization (Lasso)

- Penalty = sum of **absolute** coefficients
- Can shrink coefficients to **exactly zero**
- Performs **automatic feature selection**
- Produces sparse models
- Best when only a few features matter

> *Like a reality show elimination — weak coefficients get voted off*

## ElasticNet: Best of Both

- Combines L1 + L2 penalties
- Controlled by mixing parameter α:
  - α = 1 → Pure L1 (Lasso)
  - α = 0 → Pure L2 (Ridge)
  - 0 < α < 1 → Mix of both
- Can select groups of correlated features

## Comparison

| Property | L2 (Ridge) | L1 (Lasso) | ElasticNet |
|:---------|:-----------|:-----------|:-----------|
| Penalty | Squared coefficients | Absolute coefficients | Both |
| Feature Selection | No | Yes | Yes |
| Multicollinearity | Handles well | May pick one arbitrarily | Handles well |
| Sparse Model | No | Yes | Depends on α |

## Let's Visualize the Geometry

In [ ]:
theta = np.linspace(0, 2*np.pi, 100)
x_l2 = np.cos(theta)
y_l2 = np.sin(theta)

t = np.linspace(0, 1, 25)
x_l1 = np.concatenate([1-t, -t, -1+t, t])
y_l1 = np.concatenate([t, 1-t, -t, -1+t])

fig = make_subplots(rows=1, cols=2, subplot_titles=('L2 (Ridge) Constraint', 'L1 (Lasso) Constraint'))
fig.add_trace(go.Scatter(x=x_l2, y=y_l2, mode='lines', fill='toself', 
                         fillcolor='rgba(0,100,200,0.3)', line=dict(color='blue', width=2),
                         name='L2 Region'), row=1, col=1)
fig.add_trace(go.Scatter(x=x_l1, y=y_l1, mode='lines', fill='toself', 
                         fillcolor='rgba(200,100,0,0.3)', line=dict(color='orange', width=2),
                         name='L1 Region'), row=1, col=2)
for col in [1, 2]:
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=col)
    fig.add_vline(x=0, line_dash="dash", line_color="gray", row=1, col=col)
fig.update_layout(height=400, title_text="Geometry of Regularization Constraints", showlegend=False)
fig.update_xaxes(title_text="β₁", range=[-1.5, 1.5])
fig.update_yaxes(title_text="β₂", range=[-1.5, 1.5])
fig.show()

## Why the Geometry Matters

- L1 diamond has **corners on the axes** → solutions land on corners → zero coefficients
- L2 circle has **no corners** → solutions rarely hit exactly zero
- This is why L1 does feature selection and L2 doesn't

## In scikit-learn

```python
# No regularization (Course 2)
LogisticRegression(penalty=None)

# L2 (Ridge) — default
LogisticRegression(penalty='l2', C=1.0)

# L1 (Lasso)
LogisticRegression(penalty='l1', C=1.0, solver='saga')

# ElasticNet
LogisticRegression(penalty='elasticnet', C=1.0, solver='saga', l1_ratio=0.5)
```

## Why This Matters for Student Departure

- **Feature Selection**: Which features actually predict departure?
- **Handle Correlations**: GPA_1 and GPA_2 are correlated
- **Better Generalization**: Works on new student cohorts
- **Interpretability**: Sparser models are easier to explain to stakeholders

## Key Takeaways

| Concept | Remember |
|:--------|:---------|
| Regularization | Penalty on large coefficients |
| L2 (Ridge) | Shrinks but keeps all features |
| L1 (Lasso) | Feature selection via zeroing |
| ElasticNet | Combines both penalties |
| C parameter | Inverse of λ (small C = more regularization) |

**Next:** 2.2 Build a Regularized Logistic Regression Model